In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

from sklearn.metrics import confusion_matrix, classification_report


In [2]:
def get_split_breed_counts(dataset_path):
    train_counts = {}
    test_counts = {}

    train_path = os.path.join(dataset_path, 'train')
    test_path = os.path.join(dataset_path, 'test')

    paths = {'train': train_path, 'test': test_path}
    counts = {'train': train_counts, 'test': test_counts}

    for split in ['train', 'test']:
        split_path = paths[split]
        if not os.path.isdir(split_path):
            print(f"Error: {split.capitalize()} directory not found at '{split_path}'")
            return None, None

        breeds = [d for d in os.listdir(split_path)
                  if os.path.isdir(os.path.join(split_path, d))]

        for breed in breeds:
            breed_path = os.path.join(split_path, breed)
            num_images = len([
                f for f in os.listdir(breed_path)
                if os.path.isfile(os.path.join(breed_path, f))
            ])
            counts[split][breed] = num_images

    train_df = pd.DataFrame(list(train_counts.items()),
                            columns=['Breed', 'Count']).sort_values('Count', ascending=False)
    test_df = pd.DataFrame(list(test_counts.items()),
                           columns=['Breed', 'Count']).sort_values('Count', ascending=False)

    print("Successfully counted images for train and test sets.")
    return train_df, test_df


In [3]:
def generate_breed_distribution_graph(df, title, filename, output_dir="gg"):
    if df is None or df.empty:
        print(f"DataFrame for '{title}' is empty. Skipping graph generation.")
        return

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    plt.figure(figsize=(12, 8))
    sns.barplot(x='Count', y='Breed', data=df, palette='viridis', orient='h')
    plt.title(title, fontsize=16)
    plt.xlabel('Number of Images')
    plt.ylabel('Breed')
    plt.grid(axis='x', linestyle='--', alpha=0.7)

    for index, value in enumerate(df['Count']):
        plt.text(value, index, f' {value}', va='center')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()


In [4]:
def plot_training_history(history, filename, output_dir="gg"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

    ax1.plot(history['train_acc'])
    ax1.plot(history['val_acc'])
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend(['Train', 'Validation'])
    ax1.grid(True)

    ax2.plot(history['train_loss'])
    ax2.plot(history['val_loss'])
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend(['Train', 'Validation'])
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()


In [5]:
def plot_confusion_matrix(y_true, y_pred, class_names, filename, output_dir="gg"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()


In [6]:
def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    return all_labels, all_preds


In [7]:
def run_training_cycle(model, dataloaders, class_names,
                       criterion, optimizer, device, num_epochs=30):
    since = time.time()
    history = {'train_loss': [], 'train_acc': [],
               'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

        plot_training_history(history, f"training_epoch_{epoch+1}.png")

        y_true, y_pred = evaluate_model(model, dataloaders['val'], device)
        plot_confusion_matrix(y_true, y_pred, class_names,
                              f"confusion_epoch_{epoch+1}.png")

        print(classification_report(y_true, y_pred,
                                    target_names=class_names, zero_division=0))

    print(f"\nTraining complete in {(time.time()-since)//60:.0f} minutes")
    return model, history


In [8]:
dataset_path = r"D:\book college\semester 5\annpp001\Cattle-Buffalo-breeds.v1i.folder"

train_df, test_df = get_split_breed_counts(dataset_path)
generate_breed_distribution_graph(train_df,
                                  "Training Set Breed Distribution",
                                  "train_distribution.png")
generate_breed_distribution_graph(test_df,
                                  "Testing Set Breed Distribution",
                                  "test_distribution.png")


Successfully counted images for train and test sets.


C:\Users\shash\AppData\Local\Temp\ipykernel_15572\1494378305.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Count', y='Breed', data=df, palette='viridis', orient='h')
C:\Users\shash\AppData\Local\Temp\ipykernel_15572\1494378305.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Count', y='Breed', data=df, palette='viridis', orient='h')


In [9]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
}

image_datasets = {
    'train': datasets.ImageFolder(os.path.join(dataset_path, 'train'),
                                  data_transforms['train']),
    'val': datasets.ImageFolder(os.path.join(dataset_path, 'test'),
                                data_transforms['val'])
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=32,
                  shuffle=True, num_workers=4)
    for x in ['train', 'val']
}

class_names = image_datasets['train'].classes
num_classes = len(class_names)


Using device: cpu


In [10]:
model = models.convnext_tiny(pretrained=True)

num_ftrs = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(num_ftrs, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

model, history = run_training_cycle(
    model=model,
    dataloaders=dataloaders,
    class_names=class_names,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=30
)


C:\Users\shash\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\shash\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1`. You can also use `weights=ConvNeXt_Tiny_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 1/30
------------------------------
train Loss: 1.2887 Acc: 0.5543
val Loss: 0.8387 Acc: 0.7133
                   precision    recall  f1-score   support

              Gir       1.00      0.05      0.09        22
Holstein_Friesian       0.90      0.93      0.91        28
       Jaffrabadi       0.00      0.00      0.00         9
           Jersey       0.79      0.58      0.67        19
           Murrah       0.60      1.00      0.75        15
          Sahiwal       0.66      0.98      0.79        50

         accuracy                           0.71       143
        macro avg       0.66      0.59      0.53       143
     weighted avg       0.73      0.71      0.64       143


Epoch 2/30
------------------------------
train Loss: 0.7665 Acc: 0.7244
val Loss: 0.7054 Acc: 0.7203
                   precision    recall  f1-score   support

              Gir       0.00      0.00      0.00        22
Holstein_Friesian       0.90      0.93      0.91        28
       Jaffrabadi      

In [11]:
MODEL_PATH = "gg/convnext_tiny_best.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, MODEL_PATH)

print(f"Model saved successfully at {MODEL_PATH}")


Model saved successfully at gg/convnext_tiny_best.pth
